# BioMini Phase I Educational Materials

## Notebook 08 — Validation, Packaging and Scientific Software
### 실행되는 코드에서 Scientific Software로

Notebook 01–07에서는 BioMini Phase I의 핵심 architecture를 단계적으로 만들었습니다.

이제 마지막 질문을 다룹니다.

> 이 코드가 정말 의도대로 작동하는가?

그리고 더 중요한 질문은 다음입니다.

> 이 코드의 scientific behavior가 independent reference implementation과도 일치하는가?

이번 Notebook은 Phase I의 마지막 교육 unit입니다.

다음 내용을 통합합니다.

```text
package structure
pytest
regression testing
scientific reference validation
Biopython comparison
edge-case testing
pyproject.toml
version management
dependency management
build / install
GitHub Actions CI
final architecture
```

이 Notebook은 BioMini Phase I의 **Steps 9–11, 13–15와 Canonical Audit**을
교육용으로 재구성합니다.

---

## Learning Objectives

이 Notebook을 마치면 다음을 할 수 있어야 합니다.

1. script와 installable package의 차이를 설명한다.
2. regression test와 scientific reference validation을 구분한다.
3. pytest 기반 test의 역할을 이해한다.
4. edge case를 test로 고정해야 하는 이유를 설명한다.
5. Biopython을 independent scientific reference로 활용하는 방법을 이해한다.
6. `pyproject.toml`의 역할을 설명한다.
7. version과 dependency의 single source of truth 개념을 이해한다.
8. package build와 editable install의 의미를 이해한다.
9. GitHub Actions CI가 왜 필요한지 설명한다.
10. Phase I 전체 architecture를 하나의 scientific software system으로 설명한다.

# 1. 실행되는 코드와 Scientific Software는 다르다

Notebook에서 몇 줄의 코드가 정상적으로 실행된다고 해서
그 코드가 바로 scientific software가 되는 것은 아닙니다.

scientific software에는 최소한 다음 질문이 필요합니다.

```text
같은 입력에 같은 동작을 하는가?
잘못된 입력을 안전하게 거부하는가?
edge case가 명시되어 있는가?
다른 scientific implementation과 결과가 일치하는가?
다른 Python version에서도 동작하는가?
package로 설치 가능한가?
version과 dependency가 관리되는가?
```

즉:

```text
works once
```

와

```text
reliable scientific software
```

사이에는 큰 차이가 있습니다.

# 2. Script에서 Package로

초기 교육용 코드는 하나의 file 또는 Notebook 안에 있을 수 있습니다.

하지만 project가 커지면 책임별 module로 나누는 것이 필요합니다.

BioMini Phase I의 package structure는 다음과 같습니다.

```text
biomini/
├── core.py
├── sequence.py
├── analyzers.py
├── features.py
├── dataset.py
├── models.py
├── io.py
├── persistence.py
├── errors.py
├── logging_utils.py
└── _version.py
```

repository level에서는:

```text
biomini/
tests/
docs/
examples/
data/
.github/
pyproject.toml
README.md
```

형태를 가집니다.

# 3. Public API — `__init__.py`

사용자가 내부 module 경로를 모두 알 필요는 없습니다.

예를 들어 다음처럼 import할 수 있게 합니다.

```python
from biomini import DNA, ProteinAnalyzer
```

이것이 package의 public API입니다.

public API는 내부 file layout과 사용자-facing interface를 분리합니다.

# 4. Regression Testing

Regression test의 질문은 단순합니다.

> 이미 정상이라고 합의한 behavior가 이후 변경으로 깨지지 않았는가?

예를 들어 DNA normalization을 test한다고 합시다.

In [ ]:
def normalize_dna(sequence: str) -> str:
    return "".join(
        sequence.split()
    ).upper()


assert (
    normalize_dna(" atgc ")
    == "ATGC"
)

print(
    "simple regression check: PASS"
)

Notebook에서는 `assert`로 간단히 확인할 수 있지만,
real project에서는 `pytest` 같은 test framework를 사용합니다.

예:

```python
def test_dna_normalization():
    dna = DNA("gene", "atgc")
    assert dna.sequence == "ATGC"
```

# 5. pytest가 제공하는 것

pytest는 다음을 체계적으로 수행할 수 있게 해줍니다.

```text
test discovery
assertion reporting
exception testing
temporary files
test isolation
regression suite
CI integration
```

예를 들어 invalid DNA는 exception을 발생시켜야 합니다.

```python
with pytest.raises(ValidationError):
    DNA("bad", "ATGX")
```

이것도 specification의 일부입니다.

# 6. 무엇을 Test해야 하는가?

BioMini Phase I에서는 다음 영역을 test합니다.

```text
sequence normalization
validation
complement / reverse complement
transcription
translation
GC content
protein molecular weight
hydrophobic fraction
feature generation
FASTA read/write
serialization
model persistence
provenance
package API
```

중요한 점은 method 수만큼 test를 만드는 것이 아니라,
**behavioral contract를 test한다는 것**입니다.

# 7. Edge Case는 Test로 고정한다

Phase I Canonical Audit에서 중요한 edge case들이 발견되었습니다.

대표적인 예:

```text
A — empty translation product
B — nested schema validation
C — input validation consistency
D — exception hierarchy consistency
```

이런 문제는 문서에만 적어두면 다시 깨질 수 있습니다.

따라서 test로 고정합니다.

## 7.1 Empty Translation Product

다음은 valid computational result입니다.

```text
RNA = UAA
 ↓
Protein = ""
```

따라서 canonical policy는:

```text
DNA empty     → invalid
RNA empty     → invalid
Protein empty → valid
```

입니다.

그리고 empty Protein 분석값은:

```text
molecular_weight = 0.0
hydrophobic_fraction = 0.0
amino-acid fractions = 0.0
```

입니다.

In [ ]:
def translate_stop_only():
    return ""

protein_sequence = (
    translate_stop_only()
)

assert (
    protein_sequence
    == ""
)

print(
    "empty translation edge case: PASS"
)

## 7.2 Nested Schema Validation

다음 구조를 생각합니다.

```text
Dataset
 └── Sample
      └── FeatureSet
```

top-level schema만 맞는다고 충분하지 않습니다.

nested `FeatureSet`이 잘못된 version이면 전체 restoration을 거부해야 합니다.

즉 각 serializable object가 자신의 schema를 검증해야 합니다.

## 7.3 Input Validation

다음도 모두 regression test 대상입니다.

```text
empty FeatureSet name
empty features
non-numeric feature value
NaN feature value
invalid entity_id
wrong Sample type
wrong Dataset entry type
```

validation은 defensive programming이 아니라
domain contract의 일부입니다.

## 7.4 Exception Semantics

BioMini는 generic `ValueError`만 사용하는 대신
domain-specific exception hierarchy를 사용합니다.

```text
BioMiniError
├── ValidationError
├── SerializationError
├── ModelPersistenceError
├── FastaFormatError
└── DataError
```

왜 중요할까요?

사용자는 error의 의미에 따라 다른 대응을 할 수 있기 때문입니다.

# 8. Software Test와 Scientific Validation은 다르다

이 구분은 BioMini Phase I의 핵심입니다.

```text
Software Regression Test
= code가 우리가 설계한 대로 작동하는가?

Scientific Reference Validation
= 그 설계와 계산 결과가 independent scientific implementation과도 일치하는가?
```

둘은 같은 질문이 아닙니다.

# 9. Independent Scientific Reference — Biopython

Phase I에서는 Biopython을 independent reference implementation으로 사용합니다.

검증 대상:

```text
reverse complement
transcription
translation
protein molecular weight
FASTA parsing
```

BioMini가 Biopython을 대체하려는 것이 아니라,
작은 educational implementation을 established library와 비교하는 것입니다.

# 10. Reverse Complement Reference Check

In [ ]:
try:
    from Bio.Seq import Seq

    sequence = "ATGCCG"

    reference = str(
        Seq(sequence)
        .reverse_complement()
    )

    print(
        "Biopython reference:",
        reference
    )

except ImportError:
    print(
        "Biopython is not installed.\n"
        "Install validation dependencies with:\n"
        '    python -m pip install -e ".[validation]"'
    )

실제 canonical test는 BioMini 결과와 Biopython 결과를 직접 비교합니다.

개념:

```python
ours = DNA("dna", sequence).reverse_complement()
reference = str(Seq(sequence).reverse_complement())
assert ours == reference
```

# 11. Transcription Reference Check

canonical test의 핵심은 다음과 같습니다.

```python
ours = DNA("dna", sequence).transcribe().sequence
reference = str(Seq(sequence).transcribe())
assert ours == reference
```

즉 internal implementation이 아니라 externally observable scientific behavior를 비교합니다.

# 12. Translation Reference Check

Phase I translation policy는:

```text
frame 0
stop at first stop codon
ignore incomplete terminal codon
```

입니다.

Biopython에서는:

```python
Seq(sequence).translate(to_stop=True)
```

와 비교합니다.

이때 incomplete terminal codon에 대해 Biopython이 warning을 낼 수 있지만,
Phase I의 결과 비교는 동일한 translated sequence를 기준으로 합니다.

# 13. Protein Molecular Weight — 실제로 발견된 차이

scientific reference validation은 단순 형식적 절차가 아닙니다.

Phase I에서는 protein molecular weight를 Biopython과 비교하는 과정에서
residue mass precision 차이를 발견했습니다.

초기 구현은 교육용으로 반올림된 residue mass를 사용했지만,
reference validation 이후 canonical implementation은
Biopython/IUPACData와 정렬된 평균 residue mass를 사용하게 되었습니다.

이 사례는 매우 중요합니다.

> unit test가 모두 통과해도 scientific reference와 비교하면 discrepancy가 발견될 수 있다.

## Molecular Weight Reference Example

In [ ]:
try:
    import math
    from Bio.Seq import Seq
    from Bio.SeqUtils import molecular_weight

    sequence = "ACDEFGHIK"

    reference_mass = molecular_weight(
        Seq(sequence),
        seq_type="protein",
    )

    print(
        "Biopython molecular weight:",
        reference_mass
    )

except ImportError:
    print(
        "Biopython is not installed."
    )

canonical BioMini test에서는 다음 tolerance를 사용합니다.

```python
math.isclose(
    ours,
    reference,
    rel_tol=1e-9,
    abs_tol=1e-9
)
```

floating-point scientific calculation에서는
단순 `==`보다 tolerance-aware comparison이 적절할 수 있습니다.

# 14. FASTA Reference Validation

BioMini에는 minimal FASTA parser가 있습니다.

하지만 mature parser인 Biopython `SeqIO`와 결과를 비교합니다.

검증 포인트:

```text
record count
identifier
sequence
```

또한 BioMini Phase I에서는 FASTA header의 identifier와 description을 구분합니다.

# 15. Why Not Just Use Biopython?

좋은 질문입니다.

production biology work에서는 Biopython 같은 mature library를 사용하는 것이 자연스럽습니다.

BioMini가 selected operation을 직접 구현하는 이유는 **교육적 투명성**입니다.

```text
직접 구현
 ↓
algorithm / architecture 이해
 ↓
independent library와 비교
 ↓
scientific validation
```

즉:

```text
Biopython
= mature scientific implementation

BioMini
= educational scientific-software architecture
```

관계입니다.

# 16. Package Metadata — `pyproject.toml`

modern Python package에서는 `pyproject.toml`이 중요한 project configuration file입니다.

BioMini Phase I의 주요 내용:

```text
build system
project metadata
Python requirement
runtime dependencies
optional dependencies
package discovery
pytest configuration
coverage configuration
dynamic version
```

현재 runtime dependencies:

```text
pandas >= 2.0
scikit-learn >= 1.3
joblib >= 1.3
```

optional validation dependency:

```text
biopython >= 1.86
```

development dependencies:

```text
pytest >= 8.0
pytest-cov >= 5.0
```

이렇게 runtime / validation / development dependency를 분리합니다.

# 17. Dependency Single Source of Truth

Phase I audit에서는 dependency specification이 여러 file에 중복되는 문제를 제거했습니다.

현재 policy:

```text
pyproject.toml
      ↓
single dependency authority
```

왜 중요할까요?

예를 들어:

```text
requirements.txt says A
pyproject.toml says B
```

이면 어느 것이 진짜인지 모호해집니다.

scientific reproducibility에서도 dependency ambiguity는 위험합니다.

# 18. Version Single Source of Truth

BioMini package version은:

```text
biomini/_version.py
```

에서 관리합니다.

현재:

```python
__version__ = "0.1.0"
```

그리고 `pyproject.toml`은 dynamic lookup을 사용합니다.

```text
_version.py
   ↓
single source of truth
   ↓
pyproject.toml dynamic version
```

# 19. Package API Smoke Test

version과 public API도 test합니다.

예:

```python
import biomini

assert biomini.__version__ == "0.1.0"

dna = biomini.DNA("gene", "ATGGCC")
protein = dna.transcribe().translate()
analyzer = biomini.ProteinAnalyzer(protein)

assert protein.sequence == "MA"
assert analyzer.molecular_weight() > 0
```

이 test는 package install 후 public-facing import path가 실제로 동작하는지 확인합니다.

# 20. Editable Installation

개발 중에는 editable install을 사용합니다.

```bash
python -m pip install -e .
```

development와 validation dependency까지 포함하면:

```bash
python -m pip install -e ".[dev,validation]"
```

editable install의 장점은 source code를 수정하면
매번 package를 reinstall하지 않아도 변경사항이 반영된다는 것입니다.

# 21. Wheel Build

Python package는 wheel 형태로 build할 수 있습니다.

개념:

```text
source repository
   ↓
build
   ↓
wheel package
   ↓
install
```

Phase I RC1에서는 wheel build가 PASS 상태입니다.

이 검증은 project가 실제 installable package로 구성되어 있는지를 확인합니다.

# 22. compileall

Python source 전체가 syntax level에서 compile 가능한지도 확인합니다.

개념적으로:

```bash
python -m compileall biomini
```

이 검사는 test가 직접 import하지 않은 module의 syntax error도 발견할 수 있습니다.

# 23. Coverage

현재 Phase I RC1 validation status:

```text
43 tests passed
90% line coverage
```

coverage는 useful metric이지만:

> 100% coverage가 scientific correctness를 보장하지는 않습니다.

coverage는 **어떤 code path가 test되지 않았는지 찾는 도구**로 보는 것이 좋습니다.

# 24. Continuous Integration

local computer에서 test가 통과해도 다른 environment에서는 실패할 수 있습니다.

그래서 GitHub Actions CI를 사용합니다.

현재 BioMini는:

```text
Python 3.10
Python 3.11
Python 3.12
Python 3.13
```

에서 자동 test합니다.

# 25. CI Workflow 읽기

현재 `.github/workflows/ci.yml`의 핵심 흐름:

```text
push / pull request
      ↓
checkout repository
      ↓
set up Python
      ↓
install package + dev + validation dependencies
      ↓
pytest
      ↓
coverage
```

coverage는 Python 3.13 job에서 추가로 수행합니다.

## CI Conceptual Pseudocode

```yaml
on:
  push:
  pull_request:

matrix:
  python:
    - 3.10
    - 3.11
    - 3.12
    - 3.13

steps:
  - checkout
  - setup-python
  - install
  - pytest
  - coverage
```

CI의 의미는 단순 automation 이상입니다.

> repository의 현재 state가 여러 supported environment에서 계속 검증된다는 뜻입니다.

# 26. Current Phase I Validation Status

현재 RC1 validation baseline:

```text
pytest                     43 passed
coverage                   90%
compileall                 PASS
wheel build                PASS
editable installation      PASS
basic example              PASS
Biopython validation       PASS
GitHub Actions CI          PASS
Python 3.10                PASS
Python 3.11                PASS
Python 3.12                PASS
Python 3.13                PASS
```

# 27. Production Hardening

scientific software가 커지면서 다음 cross-cutting concerns도 필요해졌습니다.

```text
stable IDs
metadata
provenance
schema version
custom exceptions
logging policy
```

이 기능들은 눈에 띄는 biology algorithm은 아니지만,
maintainability와 reproducibility에 중요합니다.

# 28. Logging Policy

library가 application의 logging configuration을 함부로 바꾸면 안 됩니다.

canonical BioMini는 library logger에 `NullHandler`를 사용하고,
실제 logging configuration은 application이 결정하도록 합니다.

이것도 작은 framework와 reusable library의 차이를 보여주는 예입니다.

# 29. Final Phase I Architecture

Phase I 전체 구조를 다시 봅니다.

```text
Domain Layer
 ├── DomainEntity
 ├── Molecule
 ├── SequenceMolecule
 ├── NucleicAcid
 ├── DNA
 ├── RNA
 └── Protein

Analysis Layer
 ├── SequenceAnalyzer
 ├── NucleicAcidAnalyzer
 ├── DNAAnalyzer
 ├── RNAAnalyzer
 └── ProteinAnalyzer

Representation / Data
 ├── FeatureSet
 ├── Sample
 └── Dataset

Model
 └── ProteinPredictor

Persistence
 └── ModelArtifact

I/O
 └── FASTA

Cross-Cutting
 ├── Validation
 ├── Provenance
 ├── Schema
 ├── Exceptions
 ├── Testing
 ├── Packaging
 └── CI
```

# 30. End-to-End Phase I Flow

Phase I 전체를 하나의 흐름으로 표현하면:

```text
DNA
 ↓ transcription
RNA
 ↓ translation
Protein
 ↓
ProteinAnalyzer
 ↓
FeatureSet
 ↓
Sample
 ↓
Dataset
 ↓
scikit-learn model
 ↓
ProteinPredictor
 ↓
ModelArtifact
 ↓
save / load
 ↓
reproducible inference
```

그리고 이 전체를:

```text
pytest
Biopython validation
package build
GitHub Actions CI
```

가 둘러싸고 있습니다.

# 31. What BioMini Is — and Is Not

BioMini는 다음입니다.

```text
educational scientific-software framework
architecture learning project
Scientific AI foundation
SDL foundation
```

BioMini는 다음을 목표로 하지 않습니다.

```text
Biopython replacement
complete bioinformatics suite
production laboratory-control platform
```

핵심 가치는 algorithm 수가 아니라
**scientific components를 coherent architecture로 연결하는 과정**에 있습니다.

# 32. Self-Check — Phase I Concepts

다음 질문에 스스로 답할 수 있는지 확인하십시오.

```text
1. 왜 SequenceMolecule이 필요한가?
2. 왜 ProteinAnalyzer는 Protein을 상속하지 않는가?
3. 왜 FeatureSet이 필요한가?
4. 왜 feature-column order를 저장하는가?
5. 왜 model만 저장하면 충분하지 않은가?
6. provenance는 무엇을 해결하는가?
7. regression test와 scientific validation은 어떻게 다른가?
8. 왜 Biopython과 비교하는가?
9. 왜 version/dependency single source of truth가 필요한가?
10. CI는 무엇을 보장하고 무엇을 보장하지 않는가?
```

이 질문들에 답할 수 있다면 Phase I의 핵심 architecture를 이해한 것입니다.

# 33. Canonical BioMini 실행 확인

실제 BioMini package가 설치되어 있다면 간단한 end-to-end smoke test를 수행합니다.

In [ ]:
try:
    import biomini

    dna = biomini.DNA(
        "gene",
        "ATGGCC"
    )

    protein = (
        dna
        .transcribe()
        .translate()
    )

    analyzer = (
        biomini.ProteinAnalyzer(
            protein
        )
    )

    print(
        "BioMini version:",
        biomini.__version__
    )

    print(
        "Protein:",
        protein.sequence
    )

    print(
        "Molecular weight:",
        analyzer.molecular_weight()
    )

except ImportError:
    print(
        "BioMini package is not installed in this environment.\n"
        "From the repository root, run:\n"
        '    python -m pip install -e ".[dev,validation]"'
    )

# 34. Optional — 실제 Test Suite 실행

이 Notebook을 repository root에서 실행하고 있다면
terminal에서 다음 명령을 사용할 수 있습니다.

```bash
python -m pytest
```

coverage:

```bash
python -m pytest --cov=biomini --cov-report=term-missing
```

이 cell에서는 shell command를 자동 실행하지 않습니다.

교육 환경마다 working directory와 package installation 상태가 다를 수 있기 때문입니다.

# 35. Capstone Exercise — 전체 Pipeline

Phase I 전체를 직접 연결해 보십시오.

```text
DNA
 ↓
RNA
 ↓
Protein
 ↓
Analyzer
 ↓
FeatureSet
 ↓
Sample
 ↓
Dataset
 ↓
Model
 ↓
Prediction
 ↓
ModelArtifact
 ↓
Save / Restore
 ↓
Validation
```

목표는 코드를 외우는 것이 아니라,
각 layer가 왜 따로 존재하는지 설명하는 것입니다.

# 36. Phase I에서 Phase II로

Phase I은 여기서 core scientific software foundation을 완성합니다.

Phase II에서는 representation과 model이 확장됩니다.

```text
Phase I
handcrafted features
classical ML
        ↓
Phase II
k-mer
one-hot
tokenization
PyTorch
deep learning
foundation-model embeddings
```

중요한 점:

> Phase II는 Phase I을 버리고 새로 시작하는 것이 아니다.

Phase I에서 만든:

```text
domain model
analysis separation
data abstraction
persistence
provenance
testing
validation
packaging
```

위에 Scientific AI layer가 올라갑니다.

# 37. Looking Further — Phase III

Phase III에서는 software boundary가 laboratory로 확장됩니다.

```text
Goal
 ↓
Planner
 ↓
Experiment
 ↓
Instrument / Virtual Lab
 ↓
Measurement
 ↓
QC / Analysis
 ↓
Model
 ↓
Optimizer
 ↓
Next Experiment
```

Phase I의 provenance와 reproducibility가
왜 미리 필요했는지도 여기서 분명해집니다.

# 38. Final What We Learned

Phase I의 전체 교육 narrative는 다음과 같습니다.

```text
simple Python class
        ↓
biological domain object
        ↓
sequence abstraction
        ↓
biological transformation
        ↓
domain / analysis separation
        ↓
feature engineering
        ↓
dataset
        ↓
machine learning
        ↓
persistence
        ↓
provenance
        ↓
regression testing
        ↓
scientific reference validation
        ↓
packaging
        ↓
continuous integration
        ↓
scientific software foundation
```

이것이 BioMini Phase I의 핵심입니다.

> 작은 Python 예제가 어떻게 maintainable하고 testable하며
> scientifically validated된 software architecture로 발전하는가?

# 39. Exercises

## Exercise 1 — Regression Test
DNA complement에 대한 pytest test를 직접 작성하십시오.

## Exercise 2 — Exception Test
invalid DNA `"ATGX"`가 `ValidationError`를 발생시키는 test를 작성하십시오.

## Exercise 3 — Edge Case
`RNA("stop", "UAA")`가 empty Protein을 생성하는 test를 작성하십시오.

## Exercise 4 — Scientific Reference
Biopython을 이용해 reverse complement 결과를 BioMini와 비교하십시오.

## Exercise 5 — Molecular Weight
Protein `"ACDEFGHIK"`의 molecular weight를 BioMini와 Biopython에서 각각 계산하고 비교하십시오.

## Exercise 6 — FASTA
같은 FASTA file을 BioMini와 Biopython `SeqIO`로 읽고 identifier와 sequence를 비교하십시오.

## Exercise 7 — pyproject.toml
runtime dependency와 optional dependency의 차이를 설명하십시오.

## Exercise 8 — Versioning
version을 여러 file에 중복 저장하면 어떤 문제가 생길 수 있는지 설명하십시오.

## Exercise 9 — CI
왜 local test 통과만으로 충분하지 않은지 설명하십시오.

## Exercise 10 — Architecture
Phase I 전체 architecture를 직접 그림이나 text tree로 다시 구성하십시오.

## Exercise 11 — Scientific AI Bridge
Phase II에서 handcrafted features를 embedding으로 바꾸더라도 유지되는 Phase I layer를 구분하십시오.

## Exercise 12 — Final Reflection
다음 질문에 답하십시오.

> BioMini Phase I에서 가장 중요한 결과는 biological function의 개수인가,
> 아니면 architecture인가? 그 이유는 무엇인가?